In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier

# === Load full real dataset
df = pd.read_csv("Real_MIMIC.csv", low_memory=False)

# === Drop rows with missing los_seconds and binarize label
df = df.dropna(subset=["los_seconds"])
df["los_seconds"] = pd.to_numeric(df["los_seconds"], errors="coerce")
df = df.dropna(subset=["los_seconds"])
df["label"] = (df["los_seconds"] >= 345600).astype(int)

# === Drop columns with too many missing values (>90%)
df = df.loc[:, df.isnull().mean() < 0.9]

# === Separate features and labels
X = df.drop(columns=["los_seconds", "label"])
y = df["label"]

# === Encode categorical features (object/bool) and impute missing values
for col in X.select_dtypes(include=["object", "bool"]).columns:
    unique_vals = X[col].dropna().unique().tolist()
    if set(unique_vals).issubset({'True', 'False', 'true', 'false'}):
        X[col] = X[col].astype(str).map({'True': 1, 'False': 0, 'true': 1, 'false': 0})
    else:
        X[col] = LabelEncoder().fit_transform(X[col].astype(str))

# === Impute remaining NaNs (mean for numeric)
imputer = SimpleImputer(strategy='mean')
X = pd.DataFrame(imputer.fit_transform(X), columns=X.columns)

# === Train-test split (80-20)
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.2, random_state=42)

# === Train XGBoost
model = XGBClassifier(eval_metric='logloss', random_state=42)
model.fit(X_train, y_train)

# === Predict
df_test_with_preds = X_test.copy()
df_test_with_preds["true_label"] = y_test.values
df_test_with_preds["pred_xgboost"] = model.predict(X_test)

# === Save results
df_test_with_preds.to_csv("real_full_test_with_predictions.csv", index=False)
print("✅ Saved predictions to 'real_full_test_with_predictions.csv'")


✅ Saved predictions to 'real_full_test_with_predictions.csv'


In [7]:
import pandas as pd
import numpy as np

# === Load dataset ===
data = pd.read_csv("real_full_test_with_predictions.csv")

# === Parameters ===
sensitive_col = 'race'
privileged_value = 1  # WHITE
model_col = 'pred_xgboost'

# === Fairness metric function ===
def compute_fairness(y_true, y_pred, sensitive_attr, privileged_value):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    sensitive_attr = np.array(sensitive_attr)

    mask_priv = sensitive_attr == privileged_value
    mask_unpriv = ~mask_priv

    p_priv = y_pred[mask_priv].mean()
    p_unpriv = y_pred[mask_unpriv].mean()
    dpd = abs(p_priv - p_unpriv)
    di = p_unpriv / p_priv if p_priv > 0 else float('inf')

    def conf_values(y_true, y_pred):
        TP = np.sum((y_true == 1) & (y_pred == 1))
        TN = np.sum((y_true == 0) & (y_pred == 0))
        FP = np.sum((y_true == 0) & (y_pred == 1))
        FN = np.sum((y_true == 1) & (y_pred == 0))
        return TP, TN, FP, FN

    TP_p, TN_p, FP_p, FN_p = conf_values(y_true[mask_priv], y_pred[mask_priv])
    TP_u, TN_u, FP_u, FN_u = conf_values(y_true[mask_unpriv], y_pred[mask_unpriv])

    tpr_p = TP_p / (TP_p + FN_p) if TP_p + FN_p > 0 else 0
    tpr_u = TP_u / (TP_u + FN_u) if TP_u + FN_u > 0 else 0
    delta_tpr = tpr_p - tpr_u

    fpr_p = FP_p / (FP_p + TN_p) if FP_p + TN_p > 0 else 0
    fpr_u = FP_u / (FP_u + TN_u) if FP_u + TN_u > 0 else 0
    delta_fpr = fpr_p - fpr_u

    ppv_p = TP_p / (TP_p + FP_p) if TP_p + FP_p > 0 else 0
    ppv_u = TP_u / (TP_u + FP_u) if TP_u + FP_u > 0 else 0
    delta_ppv = ppv_p - ppv_u

    acc_p = (TP_p + TN_p) / (TP_p + TN_p + FP_p + FN_p) if TP_p + TN_p + FP_p + FN_p > 0 else 0
    acc_u = (TP_u + TN_u) / (TP_u + TN_u + FP_u + FN_u) if TP_u + TN_u + FP_u + FN_u > 0 else 0
    delta_acc = acc_p - acc_u

    eod = max(abs(delta_tpr), abs(delta_fpr))
    eoo = delta_tpr

    return {
        "DPD": dpd,
        "DI": di,
        "∆TPR (EoO)": eoo,
        "∆FPR": delta_fpr,
        "∆PPV": delta_ppv,
        "∆Accuracy": delta_acc,
        "EOD": eod
    }

# === Evaluate fairness for XGBoost
results = compute_fairness(
    y_true=data['true_label'],
    y_pred=data[model_col],
    sensitive_attr=data[sensitive_col],
    privileged_value=privileged_value
)

# === Convert to DataFrame and save
df_results = pd.DataFrame([{"Model": model_col, **results}])
df_results.to_csv("fairness_results_icu.csv", index=False)

print("✅ Fairness evaluation complete. Results saved as 'fairness_results_icu.csv'")
print(df_results)


✅ Fairness evaluation complete. Results saved as 'fairness_results_icu.csv'
          Model       DPD        DI  ∆TPR (EoO)      ∆FPR      ∆PPV  \
0  pred_xgboost  0.066879  0.807827    0.042581  0.026901  0.037455   

   ∆Accuracy       EOD  
0   -0.01809  0.042581  


In [9]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier

# === Load synthetic data ===
df = pd.read_csv("generated_data_CLLM_prompt_Mimic.csv")

# === Ensure los_seconds is binary (already binarized)
df = df.dropna(subset=["los_seconds"])
df["los_seconds"] = df["los_seconds"].astype(int)

# === Extract features and labels
X = df.drop(columns=["los_seconds"])
y = df["los_seconds"]

# === Convert object/bool columns to numeric
for col in X.select_dtypes(include=["object", "bool"]).columns:
    unique_vals = X[col].dropna().unique().tolist()
    if set(unique_vals).issubset({'True', 'False', 'true', 'false'}):
        X[col] = X[col].astype(str).map({'True': 1, 'False': 0, 'true': 1, 'false': 0})
    else:
        X[col] = LabelEncoder().fit_transform(X[col].astype(str))

# === Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.2, random_state=42)

# === Train XGBoost model
model = XGBClassifier(eval_metric='logloss', random_state=42)
model.fit(X_train, y_train)

# === Predict
df_test_with_preds = X_test.copy()
df_test_with_preds["true_label"] = y_test.values
df_test_with_preds["pred_xgboost"] = model.predict(X_test)

# === Save the output
df_test_with_preds.to_csv("synthetic_test_with_predictionscllmmimic.csv", index=False)
print("✅ Saved predictions to 'synthetic_test_with_predictionscllmmimic.csv'")


✅ Saved predictions to 'synthetic_test_with_predictionscllmmimic.csv'


In [10]:
import pandas as pd
import numpy as np

# === Load dataset ===
data = pd.read_csv("synthetic_test_with_predictionscllmmimic.csv")

# === Parameters ===
sensitive_col = 'race'
privileged_value = 1  # WHITE
model_col = 'pred_xgboost'

# === Fairness metric function ===
def compute_fairness(y_true, y_pred, sensitive_attr, privileged_value):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    sensitive_attr = np.array(sensitive_attr)

    mask_priv = sensitive_attr == privileged_value
    mask_unpriv = ~mask_priv

    p_priv = y_pred[mask_priv].mean()
    p_unpriv = y_pred[mask_unpriv].mean()
    dpd = abs(p_priv - p_unpriv)
    di = p_unpriv / p_priv if p_priv > 0 else float('inf')

    def conf_values(y_true, y_pred):
        TP = np.sum((y_true == 1) & (y_pred == 1))
        TN = np.sum((y_true == 0) & (y_pred == 0))
        FP = np.sum((y_true == 0) & (y_pred == 1))
        FN = np.sum((y_true == 1) & (y_pred == 0))
        return TP, TN, FP, FN

    TP_p, TN_p, FP_p, FN_p = conf_values(y_true[mask_priv], y_pred[mask_priv])
    TP_u, TN_u, FP_u, FN_u = conf_values(y_true[mask_unpriv], y_pred[mask_unpriv])

    tpr_p = TP_p / (TP_p + FN_p) if TP_p + FN_p > 0 else 0
    tpr_u = TP_u / (TP_u + FN_u) if TP_u + FN_u > 0 else 0
    delta_tpr = tpr_p - tpr_u

    fpr_p = FP_p / (FP_p + TN_p) if FP_p + TN_p > 0 else 0
    fpr_u = FP_u / (FP_u + TN_u) if FP_u + TN_u > 0 else 0
    delta_fpr = fpr_p - fpr_u

    ppv_p = TP_p / (TP_p + FP_p) if TP_p + FP_p > 0 else 0
    ppv_u = TP_u / (TP_u + FP_u) if TP_u + FP_u > 0 else 0
    delta_ppv = ppv_p - ppv_u

    acc_p = (TP_p + TN_p) / (TP_p + TN_p + FP_p + FN_p) if TP_p + TN_p + FP_p + FN_p > 0 else 0
    acc_u = (TP_u + TN_u) / (TP_u + TN_u + FP_u + FN_u) if TP_u + TN_u + FP_u + FN_u > 0 else 0
    delta_acc = acc_p - acc_u

    eod = max(abs(delta_tpr), abs(delta_fpr))
    eoo = delta_tpr

    return {
        "DPD": dpd,
        "DI": di,
        "∆TPR (EoO)": eoo,
        "∆FPR": delta_fpr,
        "∆PPV": delta_ppv,
        "∆Accuracy": delta_acc,
        "EOD": eod
    }

# === Evaluate fairness for XGBoost
results = compute_fairness(
    y_true=data['true_label'],
    y_pred=data[model_col],
    sensitive_attr=data[sensitive_col],
    privileged_value=privileged_value
)

# === Convert to DataFrame and save
df_results = pd.DataFrame([{"Model": model_col, **results}])
df_results.to_csv("fairness_results_icu_cllm.csv", index=False)

print("✅ Fairness evaluation complete. Results saved as 'fairness_results_icu.csv'")
print(df_results)


✅ Fairness evaluation complete. Results saved as 'fairness_results_icu.csv'
          Model      DPD        DI  ∆TPR (EoO)      ∆FPR      ∆PPV  ∆Accuracy  \
0  pred_xgboost  0.00586  0.991749    0.010464  0.001587 -0.002993   0.003623   

        EOD  
0  0.010464  


In [11]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier

# === Load synthetic data ===
df = pd.read_csv("generated_data_Our_prompts_MIMIC.csv")

# === Ensure los_seconds is binary (already binarized)
df = df.dropna(subset=["los_seconds"])
df["los_seconds"] = df["los_seconds"].astype(int)

# === Extract features and labels
X = df.drop(columns=["los_seconds"])
y = df["los_seconds"]

# === Convert object/bool columns to numeric
for col in X.select_dtypes(include=["object", "bool"]).columns:
    unique_vals = X[col].dropna().unique().tolist()
    if set(unique_vals).issubset({'True', 'False', 'true', 'false'}):
        X[col] = X[col].astype(str).map({'True': 1, 'False': 0, 'true': 1, 'false': 0})
    else:
        X[col] = LabelEncoder().fit_transform(X[col].astype(str))

# === Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.2, random_state=42)

# === Train XGBoost model
model = XGBClassifier(eval_metric='logloss', random_state=42)
model.fit(X_train, y_train)

# === Predict
df_test_with_preds = X_test.copy()
df_test_with_preds["true_label"] = y_test.values
df_test_with_preds["pred_xgboost"] = model.predict(X_test)

# === Save the output
df_test_with_preds.to_csv("synthetic_test_with_predictionsOurmimic.csv", index=False)
print("✅ Saved predictions to 'synthetic_test_with_predictionscllmmimic.csv'")


✅ Saved predictions to 'synthetic_test_with_predictionscllmmimic.csv'


In [ ]:
import pandas as pd
import numpy as np

# === Load dataset ===
data = pd.read_csv("synthetic_test_with_predictionsOurmimic.csv")

# === Parameters ===
sensitive_col = 'race'
privileged_value = 1  # WHITE
model_col = 'pred_xgboost'

# === Fairness metric function ===
def compute_fairness(y_true, y_pred, sensitive_attr, privileged_value):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    sensitive_attr = np.array(sensitive_attr)

    mask_priv = sensitive_attr == privileged_value
    mask_unpriv = ~mask_priv

    p_priv = y_pred[mask_priv].mean()
    p_unpriv = y_pred[mask_unpriv].mean()
    dpd = abs(p_priv - p_unpriv)
    di = p_unpriv / p_priv if p_priv > 0 else float('inf')

    def conf_values(y_true, y_pred):
        TP = np.sum((y_true == 1) & (y_pred == 1))
        TN = np.sum((y_true == 0) & (y_pred == 0))
        FP = np.sum((y_true == 0) & (y_pred == 1))
        FN = np.sum((y_true == 1) & (y_pred == 0))
        return TP, TN, FP, FN

    TP_p, TN_p, FP_p, FN_p = conf_values(y_true[mask_priv], y_pred[mask_priv])
    TP_u, TN_u, FP_u, FN_u = conf_values(y_true[mask_unpriv], y_pred[mask_unpriv])

    tpr_p = TP_p / (TP_p + FN_p) if TP_p + FN_p > 0 else 0
    tpr_u = TP_u / (TP_u + FN_u) if TP_u + FN_u > 0 else 0
    delta_tpr = tpr_p - tpr_u

    fpr_p = FP_p / (FP_p + TN_p) if FP_p + TN_p > 0 else 0
    fpr_u = FP_u / (FP_u + TN_u) if FP_u + TN_u > 0 else 0
    delta_fpr = fpr_p - fpr_u

    ppv_p = TP_p / (TP_p + FP_p) if TP_p + FP_p > 0 else 0
    ppv_u = TP_u / (TP_u + FP_u) if TP_u + FP_u > 0 else 0
    delta_ppv = ppv_p - ppv_u

    acc_p = (TP_p + TN_p) / (TP_p + TN_p + FP_p + FN_p) if TP_p + TN_p + FP_p + FN_p > 0 else 0
    acc_u = (TP_u + TN_u) / (TP_u + TN_u + FP_u + FN_u) if TP_u + TN_u + FP_u + FN_u > 0 else 0
    delta_acc = acc_p - acc_u

    eod = max(abs(delta_tpr), abs(delta_fpr))
    eoo = delta_tpr

    return {
        "DPD": dpd,
        "DI": di,
        "∆TPR (EoO)": eoo,
        "∆FPR": delta_fpr,
        "∆PPV": delta_ppv,
        "∆Accuracy": delta_acc,
        "EOD": eod
    }

# === Evaluate fairness for XGBoost
results = compute_fairness(
    y_true=data['true_label'],
    y_pred=data[model_col],
    sensitive_attr=data[sensitive_col],
    privileged_value=privileged_value
)

# === Convert to DataFrame and save
df_results = pd.DataFrame([{"Model": model_col, **results}])
df_results.to_csv("fairness_results_icu_Our.csv", index=False)

print("✅ Fairness evaluation complete. Results saved as 'fairness_results_icu.csv'")
print(df_results)


✅ Fairness evaluation complete. Results saved as 'fairness_results_icu.csv'
          Model       DPD        DI  ∆TPR (EoO)      ∆FPR      ∆PPV  \
0  pred_xgboost  0.025564  0.973686    0.018054  0.081081 -0.001536   

   ∆Accuracy       EOD  
0   0.012012  0.081081  


In [14]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier

# === Load full real dataset
df = pd.read_csv("mimic_synthetic_data_3400_samples_DECAF.csv", low_memory=False)

# === Drop rows with missing los_seconds and binarize label
df = df.dropna(subset=["los_seconds"])
df["los_seconds"] = pd.to_numeric(df["los_seconds"], errors="coerce")
df = df.dropna(subset=["los_seconds"])
df["label"] = (df["los_seconds"] >= 345600).astype(int)

# === Drop columns with too many missing values (>90%)
df = df.loc[:, df.isnull().mean() < 0.9]

# === Separate features and labels
X = df.drop(columns=["los_seconds", "label"])
y = df["label"]

# === Encode categorical features (object/bool) and impute missing values
for col in X.select_dtypes(include=["object", "bool"]).columns:
    unique_vals = X[col].dropna().unique().tolist()
    if set(unique_vals).issubset({'True', 'False', 'true', 'false'}):
        X[col] = X[col].astype(str).map({'True': 1, 'False': 0, 'true': 1, 'false': 0})
    else:
        X[col] = LabelEncoder().fit_transform(X[col].astype(str))

# === Impute remaining NaNs (mean for numeric)
imputer = SimpleImputer(strategy='mean')
X = pd.DataFrame(imputer.fit_transform(X), columns=X.columns)

# === Train-test split (80-20)
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.2, random_state=42)

# === Train XGBoost
model = XGBClassifier(eval_metric='logloss', random_state=42)
model.fit(X_train, y_train)

# === Predict
df_test_with_preds = X_test.copy()
df_test_with_preds["true_label"] = y_test.values
df_test_with_preds["pred_xgboost"] = model.predict(X_test)

# === Save results
df_test_with_preds.to_csv("DECAF_full_test_with_predictions.csv", index=False)
print("✅ Saved predictions to 'real_full_test_with_predictions.csv'")


✅ Saved predictions to 'real_full_test_with_predictions.csv'


In [15]:
import pandas as pd
import numpy as np

# === Load dataset ===
data = pd.read_csv("DECAF_full_test_with_predictions.csv")

# === Parameters ===
sensitive_col = 'race'
privileged_value = 1  # WHITE
model_col = 'pred_xgboost'

# === Fairness metric function ===
def compute_fairness(y_true, y_pred, sensitive_attr, privileged_value):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    sensitive_attr = np.array(sensitive_attr)

    mask_priv = sensitive_attr == privileged_value
    mask_unpriv = ~mask_priv

    p_priv = y_pred[mask_priv].mean()
    p_unpriv = y_pred[mask_unpriv].mean()
    dpd = abs(p_priv - p_unpriv)
    di = p_unpriv / p_priv if p_priv > 0 else float('inf')

    def conf_values(y_true, y_pred):
        TP = np.sum((y_true == 1) & (y_pred == 1))
        TN = np.sum((y_true == 0) & (y_pred == 0))
        FP = np.sum((y_true == 0) & (y_pred == 1))
        FN = np.sum((y_true == 1) & (y_pred == 0))
        return TP, TN, FP, FN

    TP_p, TN_p, FP_p, FN_p = conf_values(y_true[mask_priv], y_pred[mask_priv])
    TP_u, TN_u, FP_u, FN_u = conf_values(y_true[mask_unpriv], y_pred[mask_unpriv])

    tpr_p = TP_p / (TP_p + FN_p) if TP_p + FN_p > 0 else 0
    tpr_u = TP_u / (TP_u + FN_u) if TP_u + FN_u > 0 else 0
    delta_tpr = tpr_p - tpr_u

    fpr_p = FP_p / (FP_p + TN_p) if FP_p + TN_p > 0 else 0
    fpr_u = FP_u / (FP_u + TN_u) if FP_u + TN_u > 0 else 0
    delta_fpr = fpr_p - fpr_u

    ppv_p = TP_p / (TP_p + FP_p) if TP_p + FP_p > 0 else 0
    ppv_u = TP_u / (TP_u + FP_u) if TP_u + FP_u > 0 else 0
    delta_ppv = ppv_p - ppv_u

    acc_p = (TP_p + TN_p) / (TP_p + TN_p + FP_p + FN_p) if TP_p + TN_p + FP_p + FN_p > 0 else 0
    acc_u = (TP_u + TN_u) / (TP_u + TN_u + FP_u + FN_u) if TP_u + TN_u + FP_u + FN_u > 0 else 0
    delta_acc = acc_p - acc_u

    eod = max(abs(delta_tpr), abs(delta_fpr))
    eoo = delta_tpr

    return {
        "DPD": dpd,
        "DI": di,
        "∆TPR (EoO)": eoo,
        "∆FPR": delta_fpr,
        "∆PPV": delta_ppv,
        "∆Accuracy": delta_acc,
        "EOD": eod
    }

# === Evaluate fairness for XGBoost
results = compute_fairness(
    y_true=data['true_label'],
    y_pred=data[model_col],
    sensitive_attr=data[sensitive_col],
    privileged_value=privileged_value
)

# === Convert to DataFrame and save
df_results = pd.DataFrame([{"Model": model_col, **results}])
df_results.to_csv("fairness_results_icu_Decaf.csv", index=False)

print("✅ Fairness evaluation complete. Results saved as 'fairness_results_icu.csv'")
print(df_results)


✅ Fairness evaluation complete. Results saved as 'fairness_results_icu.csv'
          Model       DPD        DI  ∆TPR (EoO)      ∆FPR      ∆PPV  \
0  pred_xgboost  0.015186  0.952104    0.213392 -0.099914  0.221497   

   ∆Accuracy       EOD  
0    0.14174  0.213392  
